# Six-direction cross-domain sentiment evaluation

Cleaned text-only successor to `main_book_to_light.ipynb`. Despite the historical filename, it covers all six directions between Book, Game, and Movie. All, uniform averaging, and genre routing use the same full target corpora. Aggregation uses raw sentiment logits, not probabilities.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

repo = Path("/content/ro3-code") if Path("/content").exists() else Path.cwd()
if not (repo / "ro3_common.py").is_file():
    if not Path("/content").exists():
        raise RuntimeError("Run this notebook from the cloned repository root.")
    subprocess.run([
        "git", "clone",
        "https://github.com/Jun-Wang0/Domain-level-learning-and-genre-specific-expert-ensembles.git",
        str(repo),
    ], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

os.environ["RO3_ROOT"] = "/content/drive/MyDrive/PhD"
os.environ["RO3_OUTPUT"] = str(Path(os.environ["RO3_ROOT"]) / "RO3_code_release")
print("Repository:", repo)
print("New experiment outputs:", os.environ["RO3_OUTPUT"])


## Checkpoint configuration

By default, use checkpoints created by the release trainers. Set `USE_EXISTING_DRIVE_MODELS` to true to read the historical directories. For old routers without semantic label mappings, supply a JSON file containing verified original training-column orders. Do not guess the order. Uniform evaluation does not need a router or an order file.

In [ ]:
USE_EXISTING_DRIVE_MODELS = False
if USE_EXISTING_DRIVE_MODELS:
    root = Path(os.environ["RO3_ROOT"])
    os.environ["RO3_SENTIMENT_MODELS"] = str(root / "Test_Version_Model/Sentiment_model")
    os.environ["RO3_ROUTER_MODELS"] = str(root / "Test_Version_Model/Genre_model")
else:
    os.environ.pop("RO3_SENTIMENT_MODELS", None)
    os.environ.pop("RO3_ROUTER_MODELS", None)

SOURCES = ["Book", "Game", "Movie"]
TARGETS = ["Book", "Game", "Movie"]
ALLOCATION = "2+2"
AGGREGATION = "both"
BATCH_SIZE = 32
ROUTER_LABELS_JSON = None

In [ ]:
from evaluate_cross_domain import evaluate

results = evaluate(sources=SOURCES, targets=TARGETS, allocation=ALLOCATION,
                   aggregation=AGGREGATION, batch_size=BATCH_SIZE,
                   router_labels=ROUTER_LABELS_JSON)
results

Results and prediction labels are saved after each completed transfer pair under `RO3_OUTPUT/results`. Original results, data, and checkpoints are not overwritten. This notebook contains no stored experimental outputs.